# When Nyström KPCA beats PCA

Normal points lie near a unit circle; anomalies lie near its center. Linear PCA with one component gives the center a *small* reconstruction error. A quadratic kernel maps $(x_1,x_2)$ to $(x_1^2,\sqrt{2}x_1x_2,x_2^2)$: normal points vary in two directions, while the center differs in a third, radial direction. Two-component kernel PCA can therefore reconstruct the ring but not the center.

Run this notebook from the repository root using the project's Python environment. It generates everything in memory, calls the existing model adapters, and writes no dataset or run artifacts. If you need a notebook server, use `uv run --with jupyterlab --with ipykernel --with nbformat jupyter lab`. Run all cells to display the figures inline. This constructed example is a sanity check, not evidence about Credit Card.

In [ ]:
import sys
from pathlib import Path

root = next(
    path
    for path in (Path.cwd(), Path.cwd().parent)
    if (path / "src" / "mark_ad_benchmark").is_dir()
)
sys.path.insert(0, str(root / "src"))

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.metrics import average_precision_score, f1_score

from mark_ad_benchmark.logging import Logger
from mark_ad_benchmark.models import NystromKPCAModel, PCAModel
from mark_ad_benchmark.thresholds.validation_f1_quantile import (
    ValidationF1QuantileThresholdSelector,
)

pio.renderers.default = "plotly_mimetype+notebook"
rng = np.random.default_rng(42)


def make_split(n_normal, n_anomaly=0):
    angle = rng.uniform(0, 2 * np.pi, n_normal)
    radius = rng.normal(1, 0.02, n_normal)
    normal = np.column_stack((radius * np.cos(angle), radius * np.sin(angle)))
    anomaly = rng.normal(0, 0.04, (n_anomaly, 2))
    features = np.vstack((normal, anomaly))
    labels = np.r_[np.zeros(n_normal, dtype=int), np.ones(n_anomaly, dtype=int)]
    order = rng.permutation(len(labels))
    return features[order], labels[order]


train, train_labels = make_split(512)
validation, validation_labels = make_split(320, 32)
test, test_labels = make_split(320, 32)

fig = make_subplots(rows=1, cols=3, subplot_titles=("Train", "Validation", "Test"))
for col, (features, labels) in enumerate(
    ((train, train_labels), (validation, validation_labels), (test, test_labels)), 1
):
    for label, name, color in ((0, "Normal", "#4477AA"), (1, "Anomaly", "#CC3311")):
        points = features[labels == label]
        fig.add_trace(
            go.Scatter(
                x=points[:, 0],
                y=points[:, 1],
                mode="markers",
                name=name,
                marker={"color": color, "size": 5},
                showlegend=col == 2,
            ),
            row=1,
            col=col,
        )
fig.update_xaxes(range=[-1.3, 1.3], title_text="x₁")
fig.update_yaxes(range=[-1.3, 1.3], title_text="x₂")
fig.update_layout(width=1050, height=400, title="Independent seeded splits")
fig.show()

## Fit on normal training points only

Both models use the same unscaled splits. The quadratic landmark feature space has rank 3; Nyström keeps 2 principal directions for reconstruction and leaves the radial direction available as an anomaly residual. The threshold is selected on validation labels, never on test labels.

In [ ]:
logger = Logger().for_component("notebooks.synthetic_kpca_circle")
models = {
    "PCA": PCAModel({"n_components": 1, "standardize_features": False}, logger),
    "Nyström KPCA": NystromKPCAModel(
        {
            "n_components": 2,
            "n_landmarks": 64,
            "kernel": "poly",
            "kernel_params": {"gamma": 1.0, "degree": 2, "coef0": 0.0},
            "standardize_features": False,
            "random_state": 42,
            "eigenvalue_tol_mode": "relative",
        },
        logger,
    ),
}
results = {}
for name, model in models.items():
    model.fit(train)
    validation_scores = np.asarray(model.score(validation))
    test_scores = np.asarray(model.score(test))

    assert np.isfinite(validation_scores).all() and np.isfinite(test_scores).all()
    selector = ValidationF1QuantileThresholdSelector(logger)
    threshold = selector.fit(validation_scores, validation_labels)
    results[name] = {
        "validation_scores": validation_scores,
        "test_scores": test_scores,
        "validation_ap": average_precision_score(validation_labels, validation_scores),
        "test_ap": average_precision_score(test_labels, test_scores),
        "validation_f1": f1_score(validation_labels, validation_scores >= threshold),
        "test_f1": f1_score(
            test_labels, test_scores >= selector.threshold_for(test_scores)
        ),
    }

assert models["Nyström KPCA"].estimator.normalizer_.shape[1] == 3
assert results["Nyström KPCA"]["test_ap"] > results["PCA"]["test_ap"] + 0.3
assert min(results["Nyström KPCA"]["test_scores"][test_labels == 1]) > max(
    results["Nyström KPCA"]["test_scores"][test_labels == 0]
)
print("Model              Validation AP   Test AP   Validation F1   Test F1")
for name, row in results.items():
    print(
        f"{name:18} {row['validation_ap']:13.3f} {row['test_ap']:9.3f} {row['validation_f1']:15.3f} {row['test_f1']:9.3f}"
    )

## Where each model assigns high anomaly scores

The maps score a grid of new points without refitting either model. Only a $P\times M$ train–landmark kernel is used during Nyström fitting; no full $P\times P$ kernel is built.

In [ ]:
axis = np.linspace(-1.3, 1.3, 101)
xx, yy = np.meshgrid(axis, axis)
grid = np.column_stack((xx.ravel(), yy.ravel()))
fig = make_subplots(rows=1, cols=2, subplot_titles=tuple(models))
for col, model in enumerate(models.values(), 1):
    surface = np.asarray(model.score(grid)).reshape(xx.shape)
    fig.add_trace(
        go.Heatmap(
            x=axis,
            y=axis,
            z=surface,
            colorscale="Viridis",
            colorbar={"title": "Residual", "x": 0.43 if col == 1 else 1.0},
        ),
        row=1,
        col=col,
    )
    for label, color in ((0, "#FFFFFF"), (1, "#CC3311")):
        points = test[test_labels == label]
        fig.add_trace(
            go.Scatter(
                x=points[:, 0],
                y=points[:, 1],
                mode="markers",
                marker={"color": color, "size": 4},
                showlegend=False,
            ),
            row=1,
            col=col,
        )
fig.update_layout(width=950, height=440, title="Test points on held-out score maps")
fig

## Multivariate time series: missing-amplitude events

Four sensors follow a periodic cycle. In several contiguous validation and test intervals, the signal collapses toward zero. We fit on an earlier, anomaly-free period and score later periods row by row, as the benchmark does. This tests nonlinear **row** anomalies in time-series data; neither model sees neighboring rows, so it cannot detect changes in ordering or frequency alone.

In [ ]:
time_rng = np.random.default_rng(2026)
sensors = np.array(((1, 0), (0, 1), (0.7, 0.7), (0.7, -0.7)))


def make_time_split(length, start, events=()):
    ticks = np.arange(start, start + length)
    angle = 2 * np.pi * ticks / 48 + time_rng.normal(0, 0.01, length)
    amplitude = time_rng.normal(1, 0.02, length)
    labels = np.zeros(length, dtype=int)
    for begin, end in events:
        labels[begin:end] = 1
        amplitude[begin:end] = time_rng.normal(0.04, 0.005, end - begin)
    latent = np.column_stack((amplitude * np.cos(angle), amplitude * np.sin(angle)))
    features = latent @ sensors.T + time_rng.normal(0, 0.005, (length, 4))
    return ticks, features, labels


validation_events = ((95, 120), (310, 335))
test_events = ((120, 145), (335, 360))
train_time, train_series, train_labels_time = make_time_split(768, 0)
validation_time, validation_series, validation_labels_time = make_time_split(
    480, 768, validation_events
)
test_time, test_series, test_labels_time = make_time_split(480, 1248, test_events)
assert not train_labels_time.any()

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    subplot_titles=("Four test channels", "Anomaly intervals"),
)
for channel in range(4):
    fig.add_trace(
        go.Scatter(
            x=test_time,
            y=test_series[:, channel],
            name=f"Sensor {channel + 1}",
            mode="lines",
        ),
        row=1,
        col=1,
    )
fig.add_trace(
    go.Scatter(
        x=test_time, y=test_labels_time, name="Anomaly", mode="lines", line_shape="hv"
    ),
    row=2,
    col=1,
)
fig.update_layout(width=1000, height=500, title="Held-out multivariate time series")
fig

Fit fresh instances of the same two models and use validation labels to select the upper-tail detection fraction. The test labels are used only for evaluation. Shaded spans in the plot below are the known test anomaly intervals.

In [ ]:
time_models = {
    "PCA": PCAModel({"n_components": 1, "standardize_features": False}, logger),
    "Nyström KPCA": NystromKPCAModel(models["Nyström KPCA"].parameters, logger),
}
time_results = {}
for name, model in time_models.items():
    model.fit(train_series)
    validation_scores = np.asarray(model.score(validation_series))
    test_scores = np.asarray(model.score(test_series))
    assert np.isfinite(validation_scores).all() and np.isfinite(test_scores).all()
    selector = ValidationF1QuantileThresholdSelector(logger)
    validation_threshold = selector.fit(validation_scores, validation_labels_time)
    test_threshold = selector.threshold_for(test_scores)
    time_results[name] = {
        "scores": test_scores,
        "threshold": test_threshold,
        "validation_ap": average_precision_score(
            validation_labels_time, validation_scores
        ),
        "test_ap": average_precision_score(test_labels_time, test_scores),
        "validation_f1": f1_score(
            validation_labels_time, validation_scores >= validation_threshold
        ),
        "test_f1": f1_score(test_labels_time, test_scores >= test_threshold),
    }

assert time_results["Nyström KPCA"]["test_ap"] > time_results["PCA"]["test_ap"] + 0.3
print("Model              Validation AP   Test AP   Validation F1   Test F1")
for name, row in time_results.items():
    print(
        f"{name:18} {row['validation_ap']:13.3f} {row['test_ap']:9.3f} {row['validation_f1']:15.3f} {row['test_f1']:9.3f}"
    )

In [ ]:
fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    subplot_titles=("Test channels", "PCA anomaly score", "Nyström KPCA anomaly score"),
)
for channel in range(4):
    fig.add_trace(
        go.Scatter(
            x=test_time,
            y=test_series[:, channel],
            name=f"Sensor {channel + 1}",
            mode="lines",
        ),
        row=1,
        col=1,
    )
for row_number, (name, result) in enumerate(time_results.items(), 2):
    fig.add_trace(
        go.Scatter(x=test_time, y=result["scores"], name=name, mode="lines"),
        row=row_number,
        col=1,
    )
    fig.add_hline(y=result["threshold"], line_dash="dash", row=row_number, col=1)
    detected = result["scores"] >= result["threshold"]
    fig.add_trace(
        go.Scatter(
            x=test_time[detected],
            y=result["scores"][detected],
            mode="markers",
            name=f"{name} detections",
        ),
        row=row_number,
        col=1,
    )
for begin, end in test_events:
    for row_number in (1, 2, 3):
        fig.add_vrect(
            x0=test_time[begin],
            x1=test_time[end - 1] + 1,
            fillcolor="red",
            opacity=0.13,
            line_width=0,
            row=row_number,
            col=1,
        )
fig.update_xaxes(title_text="Time step", row=3, col=1)
fig.update_layout(
    width=1000,
    height=780,
    title="Held-out scores, validation-calibrated thresholds, and labeled intervals",
)
fig